# Session 7


In [ ]:
pip install pygame

In [ ]:
# refresher exercise: create a basic pygame loop 
# display one window
# make sure you can quit the window
# draw a circle on screen

In [21]:
import random
import pygame
from ugot import ugot
got = ugot.UGOT()
got.initialize("10.65.48.106") #192.168.88.1
pygame.init()
WIDTH, HEIGHT = 600, 400
screen = pygame.display.set_mode((WIDTH, HEIGHT))
font = pygame.font.SysFont(None, 32)

ROAD_WIDTH = 400
ROAD_LEFT = (WIDTH - ROAD_WIDTH) // 2
ROAD_RIGHT = ROAD_LEFT + ROAD_WIDTH
LANE_MARK_WIDTH = 8
LANE_MARK_HEIGHT = 40
LANE_MARK_SPACING = 30

CAR_WIDTH = 40
CAR_HEIGHT = 60
CAR_COLOR = (200, 30, 30)

OBSTACLE_WIDTH = 50
OBSTACLE_HEIGHT = 40
OBSTACLE_COLOR = (30, 30, 180)
OBSTACLE_SPEED = 5
SPAWN_INTERVAL = 900

MAX_X = ROAD_RIGHT - 20
MIN_X = ROAD_LEFT + 20
MAX_Y = HEIGHT - 20
MIN_Y = 20

def draw_road(scroll_offset):
    screen.fill((80, 170, 80)) # grass color
    pygame.draw.rect(screen, (40,40,40), (ROAD_LEFT,0,ROAD_WIDTH,HEIGHT))
    pygame.draw.rect(screen, (255,255,255), (ROAD_LEFT,0, 10,HEIGHT))
    pygame.draw.rect(screen, (255,255,255), (ROAD_RIGHT-10,0,10,HEIGHT))
    for y in range(-LANE_MARK_HEIGHT, HEIGHT, LANE_MARK_HEIGHT+LANE_MARK_SPACING):
        draw_y = y + scroll_offset % (LANE_MARK_HEIGHT+LANE_MARK_SPACING)
        pygame.draw.rect(screen,(255,255,255),
            (WIDTH//2-LANE_MARK_WIDTH//2, draw_y, 
            LANE_MARK_WIDTH,LANE_MARK_HEIGHT))

def draw_car(x, y):
    rect = pygame.Rect(int(x), int(y), CAR_WIDTH, CAR_HEIGHT)
    pygame.draw.rect(screen, CAR_COLOR, rect)

def draw_obstacle(obstacle):
    pygame.draw.rect(screen, OBSTACLE_COLOR, obstacle)
    pygame.draw.rect(screen, (255,255,255), obstacle, 2)

def clamp(value, low, high):
    return max(low, min(high, value))

x, y = WIDTH // 2, HEIGHT // 2
scroll_offset = 0
score = 0
center_pitch, center_roll, center_yaw, _, _, _, _, _, _ = got.read_gyro_data()
last_spawn = pygame.time.get_ticks()
obstacles = []
running = True
while running:
    score += 1
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
    pitch, roll, yaw, _, _, _, _, _, _ = got.read_gyro_data()
    move_x = (roll - center_roll) / 20
    move_y = (pitch - center_pitch) / -20

    x += move_x
    y += move_y
    x = clamp(x, MIN_X, MAX_X)
    y = clamp(y, MIN_Y, MAX_Y)

    now = pygame.time.get_ticks()
    if now - last_spawn > SPAWN_INTERVAL:
        last_spawn = now
        obs_x = random.randint(ROAD_LEFT, ROAD_RIGHT)
        obstacles.append(pygame.Rect(obs_x, -OBSTACLE_HEIGHT, 
                OBSTACLE_WIDTH, OBSTACLE_HEIGHT))

    for obs in obstacles:
        obs.y += OBSTACLE_SPEED
    
    obstacles = [obs for obs in obstacles if obs.y<HEIGHT+OBSTACLE_HEIGHT]

    player_rect = pygame.Rect(int(x), int(y), CAR_WIDTH, CAR_HEIGHT)
    if any(player_rect.colliderect(obs) for obs in obstacles):
        print(f"Game over! Your score is {score}")
        running = False

    # screen.fill((255, 255, 255))
    scroll_offset += 7
    draw_road(scroll_offset)
    for obs in obstacles:
        draw_obstacle(obs)
    # pygame.draw.circle(screen, (0,255,0), (int(x), int(y)), 20)
    draw_car(x, y)
    score_text = font.render(f"Score: {score}", True, (255,255,0))
    screen.blit(score_text, (20, 20))
    pygame.display.flip()

pygame.quit()

10.65.48.106:50051
Game over! Your score is 89
